In [1]:
!pip install ultralytics opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.6/922.6 kB 15.5 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import cv2
import locale
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split


locale.getpreferredencoding = lambda: "UTF-8"

In [3]:
def load_videos_from_folder(folder, max_frames=50):
    videos = []
    labels = []

    for label in ["Violence", "NonViolence"]:
        path = os.path.join(folder, label)
        for filename in os.listdir(path):
            if filename.endswith(".mp4") or filename.endswith(".avi"):

                filepath = os.path.join(path, filename)
                cap = cv2.VideoCapture(filepath)
                frames = []

                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame = cv2.resize(frame, (64, 64))
                    frames.append(frame)
                    if len(frames) == max_frames:
                        break
                cap.release()

                # Pad sequences to ensure they all have the same length
                while len(frames) < max_frames:
                    frames.append(np.zeros((64, 64, 3), dtype=np.uint8))

                videos.append(frames)
                labels.append(0 if label == "Violence" else 1)

    return np.array(videos), np.array(labels)

X, y = load_videos_from_folder("/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset")

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Normalize the pixel values
X_train = X_train / 255.0
X_test = X_test / 255.0

In [5]:
model = models.Sequential()

model.add(layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu'), input_shape=(20, 64, 64, 3)))
model.add(layers.TimeDistributed(layers.MaxPooling2D((2, 2))))
model.add(layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation='relu')))
model.add(layers.TimeDistributed(layers.MaxPooling2D((2, 2))))
model.add(layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu')))
model.add(layers.TimeDistributed(layers.MaxPooling2D((2, 2))))
model.add(layers.TimeDistributed(layers.Conv2D(16, (3, 3), activation='relu')))
model.add(layers.TimeDistributed(layers.MaxPooling2D((2, 2))))
model.add(layers.TimeDistributed(layers.Flatten()))

model.add(layers.LSTM(16))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
model.fit(X_train, y_train, epochs=10, batch_size=8, validation_data=(X_test, y_test))
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 57s 199ms/step - accuracy: 0.5093 - loss: 0.6942 - val_accuracy: 0.5340 - val_loss: 0.6885
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 161ms/step - accuracy: 0.5599 - loss: 0.6847 - val_accuracy: 0.5240 - val_loss: 0.6674
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 161ms/step - accuracy: 0.5769 - loss: 0.6676 - val_accuracy: 0.6320 - val_loss: 0.6285
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 161ms/step - accuracy: 0.6278 - loss: 0.6343 - val_accuracy: 0.6380 - val_loss: 0.6019
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 161ms/step - accuracy: 0.6423 - loss: 0.6087 - val_accuracy: 0.6840 - val_loss: 0.5769
Epoch 6/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 162ms/step - accuracy: 0.6779 - loss: 0.5738 - val_accuracy: 0.7060 - val_loss: 0.5427
Epoch 7/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 162ms/step - accuracy: 0.7212 - loss: 0.5448 - val_accuracy: 0.7140 - val_loss: 0.5280
Epoch 8/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 162ms/step - accuracy: 0.7445 - loss: 0

In [7]:
model.save('/kaggle/working/violence.h5')